In [ ]:
# ==========================================================
# CELL 1 — SETTINGS (change numbers here, nothing else)
# ==========================================================

HOW_MANY_DIALOGUES = 1       # how many conversations to test (max 20)
HOW_MANY_QUESTIONS_EACH = 5   # how many questions per conversation (max 20)

In [ ]:
# ==========================================================
# CELL 2 — Setup (load model once)
# ==========================================================
from brain import Brain
from agent import Agent
from tracer import Tracer
from tools import ALL_TOOLS
from memory.memory_manager import MemoryManager
import config

brain_gpu0 = Brain(TOKEN, device="cuda:0")
brain_gpu1 = Brain(TOKEN, device="cuda:1")

def make_generate_fn(brain_instance):
    def generate(prompt):
        return brain_instance.think([{"role": "user", "content": prompt}])["text"]
    return generate

local_generate_0 = make_generate_fn(brain_gpu0)
local_generate_1 = make_generate_fn(brain_gpu1)

In [ ]:
# ==========================================================
# CELL 3 — Load the dataset (plain, no groupby)
# ==========================================================
import pandas as pd

all_questions = pd.read_csv("beam_100k_questions.csv")
all_histories = pd.read_csv("beam_100k_histories.csv")

# get the list of conversation IDs, and only keep as many as we want to test
conversation_id_list = all_histories["conversation_id"].unique()
conversation_id_list = conversation_id_list[:HOW_MANY_DIALOGUES]

print("Testing", len(conversation_id_list), "conversations")

In [ ]:
# ==========================================================
# CELL 4 — Judge function (checks if the answer is correct)
# ==========================================================
import ast

def judge_abstention(agent_answer):
    answer_lower = agent_answer.lower()
    for marker in config.ABSTENTION_MARKERS:
        if marker in answer_lower:
            return True
    return False

def judge_with_rubric(agent_answer, rubric_text):
    try:
        rubric_list = ast.literal_eval(rubric_text)
    except Exception:
        rubric_list = [rubric_text]

    answer_lower = agent_answer.lower()
    matches = 0
    for point in rubric_list:
        if str(point).lower()[:20] in answer_lower:
            matches = matches + 1

    needed = max(1, len(rubric_list) // 2)
    return matches >= needed

def judge(question_type, agent_answer, rubric_text):
    if question_type == "abstention":
        return judge_abstention(agent_answer)
    else:
        return judge_with_rubric(agent_answer, rubric_text)

In [ ]:
# ==========================================================
# CELL 5 — Main loop, but split across 2 GPUs at the same time
# ==========================================================
import os
import shutil
import time
import threading

results_file = "beam_eval_results.csv"
base_folder = config.DATA_FOLDER


# --- this function does the SAME work as the old single-loop version, ---
# --- but for just HALF the conversations, using ONE specific GPU ---
def run_some_dialogues(dialogue_ids, brain_instance, generate_fn, gpu_label):

    for conversation_id in dialogue_ids:

        print("\n=== [" + gpu_label + "] Conversation:", conversation_id, "===")

        # --- give this conversation its own clean memory folder ---
        conv_folder = base_folder + "/conv_" + str(conversation_id)
        if os.path.exists(conv_folder):
            shutil.rmtree(conv_folder)
        os.makedirs(conv_folder, exist_ok=True)

        # --- IMPORTANT: set file paths directly instead of using config.DATA_FOLDER ---
        # --- (two threads changing the same global folder at once would clash) ---
        memory = MemoryManager()
        memory.historical.file_path = conv_folder + "/history.json"
        memory.semantic.file_path = conv_folder + "/facts.json"
        memory.episodic.file_path = conv_folder + "/episodes.json"

        tracer = Tracer()
        agent = Agent(brain_instance, memory, ALL_TOOLS, tracer)

        # --- get this conversation's full text and feed it into memory ---
        history_row = all_histories[all_histories["conversation_id"] == conversation_id].iloc[0]
        full_text = history_row["conversation"]

        memory.add_to_historical(full_text, generate_fn=generate_fn, auto_extract=True)

        # --- get only this conversation's questions, and only the first N of them ---
        this_conversation_questions = all_questions[all_questions["conversation_id"] == conversation_id]
        this_conversation_questions = this_conversation_questions.head(HOW_MANY_QUESTIONS_EACH)

        # --- ask each question, judge it, save the result ---
        for row_number in range(len(this_conversation_questions)):
            row = this_conversation_questions.iloc[row_number]

            question_text = row["question"]
            question_type = row["question_type"]
            rubric_text = row["rubric"]

            start_time = time.time()
            agent_answer = agent.ask(question_text)
            time_taken = time.time() - start_time

            is_correct = judge(question_type, agent_answer, rubric_text)

            print("  [" + gpu_label + "][" + question_type + "] correct =", is_correct)

            one_result = pd.DataFrame([{
                "question_id": row["question_id"],
                "conversation_id": conversation_id,
                "q_type": question_type,
                "question": question_text,
                "gold_answer": row["gold_answer"],
                "agent_answer": agent_answer,
                "correct": is_correct,
                "seconds": round(time_taken, 2),
            }])

            file_already_exists = os.path.exists(results_file)
            one_result.to_csv(results_file, index=False, mode="a", header=not file_already_exists)


# --- split the conversations into two halves, one per GPU ---
half_point = len(conversation_id_list) // 2
first_half = conversation_id_list[:half_point]
second_half = conversation_id_list[half_point:]

print("GPU 0 will do", len(first_half), "conversations")
print("GPU 1 will do", len(second_half), "conversations")

# --- start both halves running AT THE SAME TIME ---
thread_gpu0 = threading.Thread(
    target=run_some_dialogues,
    args=(first_half, brain_gpu0, local_generate_0, "GPU0")
)
thread_gpu1 = threading.Thread(
    target=run_some_dialogues,
    args=(second_half, brain_gpu1, local_generate_1, "GPU1")
)

thread_gpu0.start()
thread_gpu1.start()

# --- wait here until BOTH halves are finished ---
thread_gpu0.join()
thread_gpu1.join()

print("\nDone.")

In [ ]:
# ==========================================================
# CELL 6 — See the score
# ==========================================================
results = pd.read_csv(results_file)

print("Overall accuracy:", round(results["correct"].mean() * 100, 1), "%")
print()
print(results.groupby("q_type")["correct"].mean())